# dim_date

In [1]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

GOLD_PATH = Path("../data/gold/dimensions")
GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Generate Date Range
# ---------------------------------------------------------------------

start_date = "2022-01-01"
end_date = "2025-12-31"

date_df = pd.DataFrame({
    "date": pd.date_range(start=start_date, end=end_date)
})

# ---------------------------------------------------------------------
# Date Key
# ---------------------------------------------------------------------

date_df["date_sk"] = date_df["date"].dt.strftime("%Y%m%d").astype(int)

# ---------------------------------------------------------------------
# Date Attributes
# ---------------------------------------------------------------------

date_df["day"] = date_df["date"].dt.day

date_df["month"] = date_df["date"].dt.month

date_df["month_name"] = date_df["date"].dt.month_name()

date_df["quarter"] = "Q" + date_df["date"].dt.quarter.astype(str)

date_df["year"] = date_df["date"].dt.year

date_df["week_of_year"] = date_df["date"].dt.isocalendar().week.astype(int)

date_df["day_of_week"] = date_df["date"].dt.dayofweek + 1

date_df["day_name"] = date_df["date"].dt.day_name()

date_df["is_weekend"] = date_df["day_name"].isin(
    ["Saturday", "Sunday"]
)

# ---------------------------------------------------------------------
# Financial Year (India)
# ---------------------------------------------------------------------

def get_financial_year(date):
    if date.month >= 4:
        return f"FY{date.year}-{str(date.year + 1)[-2:]}"
    return f"FY{date.year - 1}-{str(date.year)[-2:]}"

date_df["financial_year"] = date_df["date"].apply(get_financial_year)

# ---------------------------------------------------------------------
# Reorder Columns
# ---------------------------------------------------------------------

date_df = date_df[
    [
        "date_sk",
        "date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "week_of_year",
        "day_of_week",
        "day_name",
        "is_weekend",
        "financial_year"
    ]
]

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------

date_df.to_csv(
    GOLD_PATH / "dim_date.csv",
    index=False
)

date_df.to_parquet(
    GOLD_PATH / "dim_date.parquet",
    index=False
)

print("✅ dim_date created successfully")
print(date_df.head())
print(f"\nTotal Records : {len(date_df):,}")

✅ dim_date created successfully
    date_sk       date  day  month month_name quarter  year  week_of_year  \
0  20220101 2022-01-01    1      1    January      Q1  2022            52   
1  20220102 2022-01-02    2      1    January      Q1  2022            52   
2  20220103 2022-01-03    3      1    January      Q1  2022             1   
3  20220104 2022-01-04    4      1    January      Q1  2022             1   
4  20220105 2022-01-05    5      1    January      Q1  2022             1   

   day_of_week   day_name  is_weekend financial_year  
0            6   Saturday        True      FY2021-22  
1            7     Sunday        True      FY2021-22  
2            1     Monday       False      FY2021-22  
3            2    Tuesday       False      FY2021-22  
4            3  Wednesday       False      FY2021-22  

Total Records : 1,461


# dim_channel

In [2]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Load Silver Table
# ------------------------------------------------------------------

channel = pd.read_csv(SILVER_PATH / "channel_lookup.csv")

# ------------------------------------------------------------------
# Create Surrogate Key
# ------------------------------------------------------------------

channel.insert(0, "channel_sk", range(1, len(channel) + 1))

# ------------------------------------------------------------------
# Channel Type
# ------------------------------------------------------------------

online_channels = [
    "Website",
    "Mobile App",
    "Aggregator"
]

channel["channel_type"] = channel["channel_name"].apply(
    lambda x: "Online" if x in online_channels else "Offline"
)

# ------------------------------------------------------------------
# Ownership
# ------------------------------------------------------------------

direct_channels = [
    "Website",
    "Mobile App",
    "Branch",
    "Call Center"
]

channel["ownership"] = channel["channel_name"].apply(
    lambda x: "Direct" if x in direct_channels else "Indirect"
)

# ------------------------------------------------------------------
# Digital Flag
# ------------------------------------------------------------------

digital_channels = [
    "Website",
    "Mobile App"
]

channel["is_digital"] = channel["channel_name"].isin(digital_channels)

# ------------------------------------------------------------------
# Priority
# ------------------------------------------------------------------

priority = {
    "Website": "High",
    "Mobile App": "High",
    "Agent": "High",
    "Aggregator": "Medium",
    "Partner": "Medium",
    "Branch": "Low",
    "Call Center": "Low"
}

channel["priority"] = channel["channel_name"].map(priority)

# ------------------------------------------------------------------
# Reorder Columns
# ------------------------------------------------------------------

channel = channel[
    [
        "channel_sk",
        "channel_id",
        "channel_name",
        "channel_type",
        "ownership",
        "priority",
        "is_digital"
    ]
]

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------

channel.to_csv(
    GOLD_PATH / "dim_channel.csv",
    index=False
)

channel.to_parquet(
    GOLD_PATH / "dim_channel.parquet",
    index=False
)

print("✅ dim_channel created successfully")
print(channel)

✅ dim_channel created successfully
     channel_sk  channel_id   channel_name channel_type ownership priority  \
0             1         1.0    Channel_1.0      Offline  Indirect      NaN   
1             2         2.0    Channel_2.0      Offline  Indirect      NaN   
2             3         3.0    Channel_3.0      Offline  Indirect      NaN   
3             4         4.0    Channel_4.0      Offline  Indirect      NaN   
4             5         6.0    Channel_6.0      Offline  Indirect      NaN   
..          ...         ...            ...          ...       ...      ...   
150         151       157.0  Channel_157.0      Offline  Indirect      NaN   
151         152       158.0  Channel_158.0      Offline  Indirect      NaN   
152         153       159.0  Channel_159.0      Offline  Indirect      NaN   
153         154       160.0  Channel_160.0      Offline  Indirect      NaN   
154         155       163.0  Channel_163.0      Offline  Indirect      NaN   

     is_digital  
0         

# dim_customer

In [5]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load Customer
# ---------------------------------------------------------

customer = pd.read_csv(SILVER_PATH / "customer.csv")

# ---------------------------------------------------------
# Business Key
# ---------------------------------------------------------

customer["customer_id"] = [
    f"CUST{str(i).zfill(6)}"
    for i in range(1, len(customer)+1)
]

# ---------------------------------------------------------
# Driving License Status
# ---------------------------------------------------------

customer["driving_license_status"] = customer[
    "has_driving_license"
].map({
    1: "Licensed",
    0: "No License"
})

# ---------------------------------------------------------
# Customer Segment
# ---------------------------------------------------------

customer["customer_segment"] = customer[
    "previously_insured"
].map({
    1: "Existing Customer",
    0: "New Customer"
})

# ---------------------------------------------------------
# Insurance History
# ---------------------------------------------------------

customer["insurance_history"] = customer[
    "previously_insured"
].map({
    1: "Previously Insured",
    0: "First Time Buyer"
})

# ---------------------------------------------------------
# Risk Profile
# ---------------------------------------------------------

def risk(row):

    if row["has_driving_license"] == 0:
        return "High"

    if row["customer_age"] < 25:
        return "High"

    if row["customer_age"] > 60:
        return "Medium"

    return "Low"


customer["risk_profile"] = customer.apply(
    risk,
    axis=1
)

# ---------------------------------------------------------
# Final Columns
# ---------------------------------------------------------

customer = customer[
[
"customer_sk",
"customer_id",
"gender",
"customer_age",
"age_band",
"has_driving_license",
"driving_license_status",
"previously_insured",
"insurance_history",
"customer_segment",
"region",
"risk_profile"
]
]

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

customer.to_csv(
    GOLD_PATH/"dim_customer.csv",
    index=False
)

customer.to_parquet(
    GOLD_PATH/"dim_customer.parquet",
    index=False
)

print(customer.head())
print(f"\nRows : {len(customer):,}")

   customer_sk customer_id  gender  customer_age age_band  \
0            1  CUST000001    Male            44    36-45   
1            2  CUST000002    Male            76      60+   
2            3  CUST000003    Male            47    46-60   
3            4  CUST000004    Male            21    18-25   
4            5  CUST000005  Female            29    26-35   

   has_driving_license driving_license_status  previously_insured  \
0                    1               Licensed                   0   
1                    1               Licensed                   0   
2                    1               Licensed                   0   
3                    1               Licensed                   1   
4                    1               Licensed                   1   

    insurance_history   customer_segment  region risk_profile  
0    First Time Buyer       New Customer    28.0          Low  
1    First Time Buyer       New Customer     3.0       Medium  
2    First Time Buyer     


dim_vehicle

In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# Paths
# ============================================================

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ============================================================
# Load Silver Tables
# ============================================================

vehicle = pd.read_csv(SILVER_PATH / "vehicle.csv")
safety = pd.read_csv(SILVER_PATH / "vehicle_safety.csv")

print(f"Vehicle Records : {len(vehicle):,}")
print(f"Safety Records  : {len(safety):,}")

# ============================================================
# Merge Vehicle + Safety
# ============================================================

dim_vehicle = vehicle.merge(
    safety,
    on="policy_id",
    how="left"
)

print(f"Merged Records : {len(dim_vehicle):,}")

# ============================================================
# Vehicle Age Band
# ============================================================

def vehicle_age_band(age):

    if age <= 2:
        return "New"

    elif age <= 5:
        return "Mid Age"

    else:
        return "Old"

dim_vehicle["vehicle_age_band"] = dim_vehicle["vehicle_age"].apply(
    vehicle_age_band
)

# ============================================================
# Weight Class
# ============================================================

def weight_class(weight):

    if weight < 1000:
        return "Light"

    elif weight <= 1500:
        return "Medium"

    else:
        return "Heavy"

dim_vehicle["weight_class"] = dim_vehicle["gross_weight"].apply(
    weight_class
)

# ============================================================
# Engine Size
# ============================================================

def engine_category(cc):

    if cc < 1000:
        return "Small"

    elif cc <= 1500:
        return "Medium"

    else:
        return "Large"

dim_vehicle["engine_category"] = dim_vehicle["displacement"].apply(
    engine_category
)

# ============================================================
# Safety Rating
# ============================================================

def safety_rating(score):

    if score >= 90:
        return "Excellent"

    elif score >= 75:
        return "Good"

    elif score >= 60:
        return "Average"

    return "Poor"

dim_vehicle["safety_rating"] = dim_vehicle["safety_score"].apply(
    safety_rating
)

# ============================================================
# Vehicle Size
# ============================================================

vehicle_area = (
    dim_vehicle["length"]
    * dim_vehicle["width"]
)

def vehicle_size(area):

    if area < 5000000:
        return "Compact"

    elif area < 6500000:
        return "Mid Size"

    return "Large"

dim_vehicle["vehicle_size"] = vehicle_area.apply(
    vehicle_size
)

# ============================================================
# Premium Vehicle Flag
# ============================================================

premium_brands = [
    "Skoda",
    "Volkswagen",
    "Honda",
    "Hyundai",
    "Toyota"
]

dim_vehicle["premium_vehicle"] = dim_vehicle["make"].isin(
    premium_brands
)

# ============================================================
# Transmission Type
# ============================================================

dim_vehicle["transmission_category"] = dim_vehicle[
    "transmission"
].replace({

    "Automatic": "Automatic",

    "Manual": "Manual"

})

# ============================================================
# Reorder Columns
# ============================================================

dim_vehicle = dim_vehicle[
[
"vehicle_sk",
"policy_id",

"make",
"model",
"segment",

"fuel_type",

"vehicle_age",
"vehicle_age_band",

"engine_type",
"displacement",
"engine_category",

"cylinder",

"transmission",
"transmission_category",

"steering",

"length",
"width",
"height",

"gross_weight",
"weight_class",

"vehicle_size",

"airbags",
"is_esc",
"is_tpms",
"is_parking_sensors",
"is_parking_camera",
"is_brake_assist",
"is_power_steering",
"is_speed_alert",

"ncap_rating",
"safety_score",
"safety_rating",

"premium_vehicle"
]
]

# ============================================================
# Save
# ============================================================

dim_vehicle.to_csv(
    GOLD_PATH / "dim_vehicle.csv",
    index=False
)

dim_vehicle.to_parquet(
    GOLD_PATH / "dim_vehicle.parquet",
    index=False
)

print("\n=======================================")
print("dim_vehicle Created Successfully")
print("=======================================\n")

print(dim_vehicle.head())

print(f"\nRows    : {len(dim_vehicle):,}")
print(f"Columns : {len(dim_vehicle.columns)}")

Vehicle Records : 97,655
Safety Records  : 97,655
Merged Records : 97,655

dim_vehicle Created Successfully

   vehicle_sk policy_id  make model segment fuel_type  vehicle_age  \
0           1   ID00001     1    M1       A       CNG         0.05   
1           2   ID00002     1    M1       A       CNG         0.02   
2           3   ID00003     1    M1       A       CNG         0.02   
3           4   ID00004     1    M2      C1    Petrol         0.11   
4           5   ID00005     2    M3       A    Petrol         0.11   

  vehicle_age_band         engine_type  displacement  ... is_tpms  \
0              New   F8D Petrol Engine           796  ...     NaN   
1              New   F8D Petrol Engine           796  ...     NaN   
2              New   F8D Petrol Engine           796  ...     NaN   
3              New  1.2 L K12N Dualjet          1197  ...     NaN   
4              New             1.0 SCe           999  ...     NaN   

   is_parking_sensors is_parking_camera is_brake_assist

dim_policy

In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# Paths
# ============================================================

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ============================================================
# Load Policy
# ============================================================

policy = pd.read_csv(SILVER_PATH / "policy.csv")

print(f"Policy Records : {len(policy):,}")

# ============================================================
# Tenure Category
# ============================================================

def tenure_category(x):

    if x < 1:
        return "Short Term"

    elif x <= 2:
        return "Standard"

    else:
        return "Long Term"

policy["tenure_category"] = policy["policy_tenure"].apply(
    tenure_category
)

# ============================================================
# Active Flag
# ============================================================

policy["active_flag"] = policy["policy_status"].apply(
    lambda x: True if str(x).strip().lower() == "active" else False
)

# ============================================================
# Renewal Probability
# ============================================================

def renewal_probability(row):

    if row["policy_status"] == "Active":

        if row["policy_tenure"] >= 2:
            return "High"

        return "Medium"

    return "Low"

policy["renewal_probability"] = policy.apply(
    renewal_probability,
    axis=1
)

# ============================================================
# Policy Age Score
# ============================================================

def policy_age_score(x):

    if x < 1:
        return 1

    elif x <= 2:
        return 2

    else:
        return 3

policy["policy_age_score"] = policy["policy_tenure"].apply(
    policy_age_score
)

# ============================================================
# Reorder Columns
# ============================================================

policy = policy[
[
    "policy_sk",
    "policy_id",

    "policy_type",
    "coverage_type",

    "policy_status",
    "active_flag",

    "policy_tenure",
    "policy_tenure_band",
    "tenure_category",

    "renewal_probability",
    "policy_age_score",

    "source_system",
    "load_date"
]
]

# ============================================================
# Save
# ============================================================

policy.to_csv(
    GOLD_PATH / "dim_policy.csv",
    index=False
)

policy.to_parquet(
    GOLD_PATH / "dim_policy.parquet",
    index=False
)

print("\n===================================")
print("dim_policy Created Successfully")
print("===================================\n")

print(policy.head())

print(f"\nRows    : {len(policy):,}")
print(f"Columns : {len(policy.columns)}")

Policy Records : 97,655

dim_policy Created Successfully

   policy_sk policy_id    policy_type coverage_type policy_status  \
0          1   ID00001  Comprehensive          Gold        Active   
1          2   ID00002  Comprehensive         Basic       Expired   
2          3   ID00003  Comprehensive          Gold        Active   
3          4   ID00004  Comprehensive         Basic        Active   
4          5   ID00005    Third Party         Basic        Active   

   active_flag  policy_tenure policy_tenure_band tenure_category  \
0         True       0.515874        6-12 Months      Short Term   
1        False       0.672619        6-12 Months      Short Term   
2         True       0.841110        6-12 Months      Short Term   
3         True       0.900277        6-12 Months      Short Term   
4         True       0.596403        6-12 Months      Short Term   

  renewal_probability  policy_age_score     source_system  \
0              Medium                 1  Insurance Claims

# Phase 2

fact_quote

In [7]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold")

FACT_PATH = GOLD_PATH / "facts"
DIM_PATH = GOLD_PATH / "dimensions"

FACT_PATH.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

quote = pd.read_csv(SILVER_PATH / "quote.csv")
channel = pd.read_csv(DIM_PATH / "dim_channel.csv")

print(f"Quote Records   : {len(quote):,}")
print(f"Channel Records : {len(channel):,}")

# ==========================================================
# Data Type Alignment
# ==========================================================

quote["sales_channel"] = quote["sales_channel"].astype(int)
channel["channel_id"] = channel["channel_id"].astype(int)

quote["quote_date"] = pd.to_datetime(quote["quote_date"])

# ==========================================================
# Create Date Key
# ==========================================================

quote["date_sk"] = quote["quote_date"].dt.strftime("%Y%m%d").astype(int)

# ==========================================================
# Join Channel Dimension
# ==========================================================

quote = quote.merge(
    channel[["channel_sk", "channel_id", "channel_name"]],
    left_on="sales_channel",
    right_on="channel_id",
    how="left"
)

# ==========================================================
# Business Columns
# ==========================================================

quote["conversion_flag"] = quote["accepted_offer"].astype(int)

quote["quote_value_band"] = pd.cut(
    quote["quoted_premium"],
    bins=[0, 5000, 10000, 20000, float("inf")],
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

# ==========================================================
# Final Fact Table
# ==========================================================

fact_quote = quote[
[
    "quote_sk",
    "quote_id",

    "customer_sk",

    "channel_sk",
    "channel_id",
    "channel_name",

    "date_sk",

    "quoted_premium",

    "accepted_offer",
    "conversion_flag",

    "quote_status",
    "quote_stage",

    "device_type",
    "quote_source",

    "quote_value_band"
]
]

# ==========================================================
# Save
# ==========================================================

fact_quote.to_csv(
    FACT_PATH / "fact_quote.csv",
    index=False
)

fact_quote.to_parquet(
    FACT_PATH / "fact_quote.parquet",
    index=False
)

print("\n========================================")
print("fact_quote Created Successfully")
print("========================================")

print(fact_quote.head())

print(f"\nRows    : {len(fact_quote):,}")
print(f"Columns : {len(fact_quote.columns)}")

Quote Records   : 381,109
Channel Records : 155

fact_quote Created Successfully
   quote_sk quote_id  customer_sk  channel_sk  channel_id   channel_name  \
0         1  Q100000            1          25          26   Channel_26.0   
1         2  Q100001            2          25          26   Channel_26.0   
2         3  Q100002            3          25          26   Channel_26.0   
3         4  Q100003            4         146         152  Channel_152.0   
4         5  Q100004            5         146         152  Channel_152.0   

    date_sk  quoted_premium  accepted_offer  conversion_flag quote_status  \
0  20250111          5482.0               1                1    Converted   
1  20250505          5705.0               0                0    Abandoned   
2  20250920          6300.0               1                1    Converted   
3  20250527          3940.0               0                0    Abandoned   
4  20251119          3936.0               0                0    Abandoned   


fact_claim

In [4]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

SILVER = Path("../data/silver")
GOLD = Path("../data/gold")

FACT_PATH = GOLD / "facts"
DIM_PATH = GOLD / "dimensions"

# ==========================================================
# Read Tables
# ==========================================================

claim = pd.read_csv(SILVER / "claim.csv")

policy = pd.read_csv(DIM_PATH / "dim_policy.csv")

# ==========================================================
# Join Policy Dimension
# ==========================================================

claim = claim.merge(

    policy[["policy_sk","policy_id"]],

    on="policy_id",

    how="left"

)

# ==========================================================
# Claim Approved Flag
# ==========================================================

claim["claim_approved"] = claim["claim_flag"]

# ==========================================================
# Settlement Band
# ==========================================================

def settlement_band(days):

    if days <= 3:
        return "Fast"

    elif days <= 7:
        return "Standard"

    return "Delayed"

claim["settlement_band"] = claim["settlement_days"].apply(
    settlement_band
)

# ==========================================================
# Final Fact
# ==========================================================

fact_claim = claim[
[
"claim_sk",
"claim_id",

"policy_sk",

"claim_amount",

"claim_flag",

"claim_approved",

"claim_severity",

"fraud_risk",

"settlement_days",

"settlement_band"
]
]

# ==========================================================
# Save
# ==========================================================

fact_claim.to_csv(
    FACT_PATH/"fact_claim.csv",
    index=False
)

fact_claim.to_parquet(
    FACT_PATH/"fact_claim.parquet",
    index=False
)

print(fact_claim.head())

print("\nRows :",len(fact_claim))

   claim_sk   claim_id  policy_sk  claim_amount  claim_flag  claim_approved  \
0         1  CLM100000          1             0         0.0             0.0   
1         2  CLM100001          2             0         0.0             0.0   
2         3  CLM100002          3             0         0.0             0.0   
3         4  CLM100003          4             0         0.0             0.0   
4         5  CLM100004          5             0         0.0             0.0   

  claim_severity fraud_risk  settlement_days settlement_band  
0            Low        Low                0            Fast  
1            Low        Low                0            Fast  
2            Low        Low                0            Fast  
3            Low        Low                0            Fast  
4            Low        Low                0            Fast  

Rows : 97655


fact_underwriting (synthetic)

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timedelta

# ==========================================================
# Configuration
# ==========================================================

np.random.seed(42)

GOLD_PATH = Path("../data/gold")
DIM_PATH = GOLD_PATH / "dimensions"
FACT_PATH = GOLD_PATH / "facts"

FACT_PATH.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Dimensions
# ==========================================================

customer = pd.read_csv(DIM_PATH / "dim_customer.csv")
policy = pd.read_csv(DIM_PATH / "dim_policy.csv")
vehicle = pd.read_csv(DIM_PATH / "dim_vehicle.csv")
date = pd.read_csv(DIM_PATH / "dim_date.csv")

# ==========================================================
# Create Base Dataset
# ==========================================================

rows = min(
    len(policy),
    len(customer),
    len(vehicle)
)

underwriting = pd.DataFrame()

underwriting["underwriting_sk"] = range(1, rows + 1)

underwriting["policy_sk"] = policy["policy_sk"].iloc[:rows].values
underwriting["customer_sk"] = customer["customer_sk"].iloc[:rows].values
underwriting["vehicle_sk"] = vehicle["vehicle_sk"].iloc[:rows].values

# ==========================================================
# Random Underwriting Date
# ==========================================================

sample_dates = np.random.choice(
    date["date_sk"],
    rows
)

underwriting["underwriting_date_sk"] = sample_dates

# ==========================================================
# AI Risk Score
# ==========================================================

underwriting["risk_score"] = np.random.randint(
    5,
    100,
    rows
)

# ==========================================================
# AI Confidence
# ==========================================================

underwriting["ai_confidence"] = np.round(
    np.random.uniform(
        75,
        99.9,
        rows
    ),
    2
)

# ==========================================================
# Fraud Probability
# ==========================================================

underwriting["fraud_probability"] = np.round(
    underwriting["risk_score"] / 100,
    2
)

# ==========================================================
# Decision Logic
# ==========================================================

def decision(score):

    if score < 35:
        return "Approved"

    elif score < 70:
        return "Manual Review"

    return "Rejected"

underwriting["underwriting_decision"] = underwriting[
    "risk_score"
].apply(decision)

# ==========================================================
# Manual Review
# ==========================================================

underwriting["manual_review_flag"] = underwriting[
    "underwriting_decision"
].eq("Manual Review")

# ==========================================================
# Review Time
# ==========================================================

def review_time(row):

    if row == "Approved":
        return np.random.randint(2,8)

    if row == "Manual Review":
        return np.random.randint(20,90)

    return np.random.randint(5,25)

underwriting["review_time_minutes"] = underwriting[
    "underwriting_decision"
].apply(review_time)

# ==========================================================
# Premium Adjustment
# ==========================================================

def premium_adj(score):

    if score < 35:
        return np.random.randint(-10,5)

    elif score < 70:
        return np.random.randint(0,15)

    return np.random.randint(15,35)

underwriting["premium_adjustment_pct"] = underwriting[
    "risk_score"
].apply(premium_adj)

# ==========================================================
# Rules Triggered
# ==========================================================

underwriting["rules_triggered"] = np.random.randint(
    1,
    8,
    rows
)

# ==========================================================
# Model Version
# ==========================================================

versions = [
    "UW_AI_v1.0",
    "UW_AI_v1.1",
    "UW_AI_v2.0"
]

underwriting["model_version"] = np.random.choice(
    versions,
    rows,
    p=[0.25,0.35,0.40]
)

# ==========================================================
# Underwriter
# ==========================================================

underwriting["underwriter"] = np.where(
    underwriting["manual_review_flag"],
    np.random.choice(
        [
            "Alice",
            "John",
            "David",
            "Priya",
            "Rahul"
        ],
        rows
    ),
    "AI Engine"
)

# ==========================================================
# Processing Mode
# ==========================================================

underwriting["processing_mode"] = np.where(
    underwriting["manual_review_flag"],
    "Manual",
    "Straight Through Processing"
)

# ==========================================================
# SLA Status
# ==========================================================

underwriting["sla_status"] = np.where(
    underwriting["review_time_minutes"] <= 30,
    "Within SLA",
    "SLA Breached"
)

# ==========================================================
# Explainability Flag
# ==========================================================

underwriting["explanation_generated"] = True

# ==========================================================
# Risk Band
# ==========================================================

def risk_band(score):

    if score < 30:
        return "Low"

    elif score < 60:
        return "Medium"

    elif score < 80:
        return "High"

    return "Very High"

underwriting["risk_band"] = underwriting[
    "risk_score"
].apply(risk_band)

# ==========================================================
# Recommendation
# ==========================================================

def recommendation(score):

    if score < 35:
        return "Auto Approve"

    elif score < 70:
        return "Refer Underwriter"

    return "Reject"

underwriting["recommendation"] = underwriting[
    "risk_score"
].apply(recommendation)

# ==========================================================
# Final Column Order
# ==========================================================

underwriting = underwriting[
[
"underwriting_sk",

"policy_sk",
"customer_sk",
"vehicle_sk",

"underwriting_date_sk",

"risk_score",
"risk_band",

"fraud_probability",

"underwriting_decision",
"recommendation",

"manual_review_flag",

"review_time_minutes",

"premium_adjustment_pct",

"rules_triggered",

"ai_confidence",

"processing_mode",

"underwriter",

"sla_status",

"explanation_generated",

"model_version"
]
]

# ==========================================================
# Save
# ==========================================================

underwriting.to_csv(
    FACT_PATH/"fact_underwriting.csv",
    index=False
)

underwriting.to_parquet(
    FACT_PATH/"fact_underwriting.parquet",
    index=False
)

print("="*60)
print("fact_underwriting Created Successfully")
print("="*60)

print(underwriting.head())

print("\nRows :", len(underwriting))
print("Columns :", len(underwriting.columns))

fact_underwriting Created Successfully
   underwriting_sk  policy_sk  customer_sk  vehicle_sk  underwriting_date_sk  \
0                1          1            1           1              20250131   
1                2          2            2           2              20251230   
2                3          3            3           3              20240510   
3                4          4            4           4              20250718   
4                5          5            5           5              20250204   

   risk_score risk_band  fraud_probability underwriting_decision  \
0          60      High               0.60         Manual Review   
1          32    Medium               0.32              Approved   
2          27       Low               0.27              Approved   
3          35    Medium               0.35         Manual Review   
4          70      High               0.70              Rejected   

      recommendation  manual_review_flag  review_time_minutes  \
0  Ref

fact_customer_journey (synthetic)

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import uuid

# ==========================================================
# Configuration
# ==========================================================

np.random.seed(42)

GOLD_PATH = Path("../data/gold")
DIM_PATH = GOLD_PATH / "dimensions"
FACT_PATH = GOLD_PATH / "facts"

FACT_PATH.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

quote = pd.read_csv(FACT_PATH / "fact_quote.csv")
customer = pd.read_csv(DIM_PATH / "dim_customer.csv")
date = pd.read_csv(DIM_PATH / "dim_date.csv")

# ==========================================================
# Journey Stages
# ==========================================================

journey_flow = [
    ("Website Visit", "Visited Website"),
    ("Quote Started", "Started Quote"),
    ("Vehicle Details", "Entered Vehicle Details"),
    ("KYC Upload", "Uploaded Documents"),
    ("Premium Review", "Viewed Premium"),
    ("Payment", "Payment Attempt"),
    ("Policy Issued", "Policy Generated")
]

drop_reasons = [
    "Premium Too High",
    "Documents Missing",
    "Technical Error",
    "Changed Mind",
    "Slow Response",
    "Switched Insurer",
    None
]

devices = [
    "Desktop",
    "Mobile",
    "Tablet"
]

sources = [
    "Google",
    "Facebook",
    "Direct",
    "Email",
    "Partner",
    "Organic Search"
]

# ==========================================================
# Generate Journey Events
# ==========================================================

records = []
journey_sk = 1

for _, row in quote.iterrows():

    customer_sk = row["customer_sk"]
    quote_sk = row["quote_sk"]

    session = str(uuid.uuid4())[:12]

    device = np.random.choice(
        devices,
        p=[0.30,0.60,0.10]
    )

    source = np.random.choice(sources)

    ai_used = np.random.choice(
        [True, False],
        p=[0.45,0.55]
    )

    converted = bool(row["accepted_offer"])

    if converted:
        completed_stage = len(journey_flow)
    else:
        completed_stage = np.random.randint(2,7)

    for stage_no, (stage, action) in enumerate(journey_flow, start=1):

        completed = stage_no <= completed_stage

        abandoned = (
            (not converted)
            and
            (stage_no == completed_stage)
        )

        duration = np.random.randint(
            15,
            240
        )

        if abandoned:
            exit_reason = np.random.choice(drop_reasons[:-1])
        else:
            exit_reason = None

        date_sk = np.random.choice(
            date["date_sk"]
        )

        records.append({

            "journey_sk": journey_sk,

            "customer_sk": customer_sk,

            "quote_sk": quote_sk,

            "date_sk": date_sk,

            "session_id": session,

            "stage_sequence": stage_no,

            "stage_name": stage,

            "customer_action": action,

            "duration_seconds": duration,

            "completed_flag": completed,

            "abandoned_flag": abandoned,

            "exit_reason": exit_reason,

            "device_type": device,

            "traffic_source": source,

            "ai_assistance_used": ai_used,

            "conversion_flag": converted

        })

        journey_sk += 1

        if abandoned:
            break

# ==========================================================
# Create DataFrame
# ==========================================================

fact_customer_journey = pd.DataFrame(records)

# ==========================================================
# Stage Duration Band
# ==========================================================

fact_customer_journey["duration_band"] = pd.cut(

    fact_customer_journey["duration_seconds"],

    bins=[0,30,60,120,300],

    labels=[
        "Very Fast",
        "Fast",
        "Average",
        "Slow"
    ]

)

# ==========================================================
# Funnel Status
# ==========================================================

fact_customer_journey["journey_status"] = np.where(

    fact_customer_journey["abandoned_flag"],

    "Abandoned",

    np.where(

        fact_customer_journey["conversion_flag"],

        "Converted",

        "In Progress"

    )

)

# ==========================================================
# Save
# ==========================================================

fact_customer_journey.to_csv(
    FACT_PATH / "fact_customer_journey.csv",
    index=False
)

fact_customer_journey.to_parquet(
    FACT_PATH / "fact_customer_journey.parquet",
    index=False
)

print("="*60)
print("fact_customer_journey Created Successfully")
print("="*60)

print(fact_customer_journey.head())

print(f"\nRows : {len(fact_customer_journey):,}")
print(f"Columns : {len(fact_customer_journey.columns)}")

fact_customer_journey Created Successfully
   journey_sk  customer_sk  quote_sk   date_sk    session_id  stage_sequence  \
0           1            1         1  20241110  951e7025-818               1   
1           2            1         1  20220502  951e7025-818               2   
2           3            1         1  20250523  951e7025-818               3   
3           4            1         1  20220329  951e7025-818               4   
4           5            1         1  20250128  951e7025-818               5   

        stage_name          customer_action  duration_seconds  completed_flag  \
0    Website Visit          Visited Website                86            True   
1    Quote Started            Started Quote               117            True   
2  Vehicle Details  Entered Vehicle Details               225            True   
3       KYC Upload       Uploaded Documents                89            True   
4   Premium Review           Viewed Premium               131          

fact_ai_interaction (synthetic)

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import uuid
import random

# ==========================================================
# Configuration
# ==========================================================

np.random.seed(42)
random.seed(42)

GOLD_PATH = Path("../data/gold")
FACT_PATH = GOLD_PATH / "facts"
DIM_PATH = GOLD_PATH / "dimensions"

FACT_PATH.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

quote = pd.read_csv(FACT_PATH / "fact_quote.csv")
customer = pd.read_csv(DIM_PATH / "dim_customer.csv")
date = pd.read_csv(DIM_PATH / "dim_date.csv")

# ==========================================================
# Question Categories
# ==========================================================

question_bank = {

    "Premium": [

        "Why is my premium so high?",
        "Can I reduce my premium?",
        "How is premium calculated?",
        "Why did my premium increase?"

    ],

    "Documents": [

        "Which documents are required?",
        "How do I upload RC?",
        "Is Aadhaar mandatory?",
        "Can I upload later?"

    ],

    "Claims": [

        "How do I file a claim?",
        "Is bumper damage covered?",
        "How long does claim settlement take?",
        "What is cashless claim?"

    ],

    "Policy": [

        "Can I renew online?",
        "How do I download policy?",
        "How do I cancel policy?",
        "When does policy expire?"

    ],

    "Payment": [

        "Payment failed",
        "Can I pay later?",
        "Available payment methods?",
        "Can I use EMI?"

    ]

}

# ==========================================================
# AI Responses
# ==========================================================

responses = {

    "Premium":"Premium depends on vehicle, driver profile, location and claim history.",

    "Documents":"Please upload RC, Driving License, Aadhaar and previous policy copy.",

    "Claims":"You can register a claim online or through our 24x7 support.",

    "Policy":"Policy services are available through our website and mobile application.",

    "Payment":"We support UPI, Cards, Net Banking and Wallet payments."

}

# ==========================================================
# Generate AI Interaction Records
# ==========================================================

records = []

interaction_sk = 1

for _, row in quote.iterrows():

    interactions = np.random.randint(1,4)

    for i in range(interactions):

        category = random.choice(list(question_bank.keys()))

        query = random.choice(
            question_bank[category]
        )

        confidence = round(
            np.random.uniform(0.80,0.99),
            2
        )

        response_time = np.random.randint(
            400,
            2500
        )

        escalation = np.random.choice(
            [True,False],
            p=[0.10,0.90]
        )

        resolved = not escalation

        rating = np.random.choice(
            [3,4,5],
            p=[0.15,0.35,0.50]
        )

        token_count = np.random.randint(
            120,
            950
        )

        sentiment = np.random.choice(

            [
                "Positive",
                "Neutral",
                "Negative"
            ],

            p=[0.60,0.25,0.15]

        )

        intent = category

        ai_version = np.random.choice(

            [
                "GPT-4",
                "GPT-4.1",
                "GPT-5"
            ],

            p=[0.20,0.30,0.50]

        )

        records.append({

            "interaction_sk": interaction_sk,

            "customer_sk": row["customer_sk"],

            "quote_sk": row["quote_sk"],

            "date_sk": np.random.choice(
                date["date_sk"]
            ),

            "session_id": str(uuid.uuid4()),

            "interaction_timestamp": pd.Timestamp.now(),

            "question_category": category,

            "customer_intent": intent,

            "user_query": query,

            "ai_response": responses[category],

            "response_time_ms": response_time,

            "confidence_score": confidence,

            "token_count": token_count,

            "escalation_required": escalation,

            "resolved_flag": resolved,

            "customer_rating": rating,

            "customer_sentiment": sentiment,

            "ai_model_version": ai_version

        })

        interaction_sk += 1

# ==========================================================
# Create DataFrame
# ==========================================================

fact_ai_interaction = pd.DataFrame(records)

# ==========================================================
# Response Speed Band
# ==========================================================

fact_ai_interaction["response_speed"] = pd.cut(

    fact_ai_interaction["response_time_ms"],

    bins=[0,700,1200,2000,5000],

    labels=[
        "Excellent",
        "Good",
        "Average",
        "Slow"
    ]

)

# ==========================================================
# AI Quality Score
# ==========================================================

fact_ai_interaction["ai_quality_score"] = (
    fact_ai_interaction["confidence_score"]*100*0.7
    +
    fact_ai_interaction["customer_rating"]*20*0.3
).round(2)

# ==========================================================
# Save
# ==========================================================

fact_ai_interaction.to_csv(
    FACT_PATH / "fact_ai_interaction.csv",
    index=False
)

fact_ai_interaction.to_parquet(
    FACT_PATH / "fact_ai_interaction.parquet",
    index=False
)

print("="*70)
print("fact_ai_interaction Created Successfully")
print("="*70)

print(fact_ai_interaction.head())

print(f"\nRows : {len(fact_ai_interaction):,}")
print(f"Columns : {len(fact_ai_interaction.columns)}")

fact_ai_interaction Created Successfully
   interaction_sk  customer_sk  quote_sk   date_sk  \
0               1            1         1  20220329   
1               2            1         1  20240813   
2               3            1         1  20231201   
3               4            2         2  20220121   
4               5            2         2  20240616   

                             session_id      interaction_timestamp  \
0  d427bf45-9dec-470f-8fc7-24d851160198 2026-07-31 16:27:17.193097   
1  3d143a33-b69e-4fe2-9c92-e4352f8eeee0 2026-07-31 16:27:17.193344   
2  9ba18b81-cfec-4f0d-b874-b69bc34d5ebb 2026-07-31 16:27:17.193508   
3  ff03f65f-cc22-4c2a-9eb2-19f2d591147f 2026-07-31 16:27:17.193765   
4  7fa04ced-df0d-481f-b250-8d41ea2e7ff3 2026-07-31 16:27:17.193945   

  question_category customer_intent                  user_query  \
0           Premium         Premium  Why is my premium so high?   
1            Claims          Claims   Is bumper damage covered?   
2         Do